# 의원 표시과목 구성 변화 분석 — 전국·지역·서울 (2022Q2–2026Q2)

## 분석 목적
2022년 2분기부터 2026년 2분기까지 전국 의원(1차 의료기관)의 **표시과목별 기관 수 변화**를 확인하고, 그 변화가 **지역별로 어떻게 다르게 나타났는지**를 기술한다. 핵심 질문은 다음과 같다.

1. 전체 의원 수가 늘어나는 동안 어떤 표시과목이 빠르게 늘고 어떤 과목이 정체했는가?
2. 1차의료 핵심과의 비중은 지역별로 어떻게 변화했는가?
3. 그 변화는 '지방일수록 크다'는 통념과 일치하는가?

본 분석은 심평원 '지역별 의원 표시과목별 기관 수' 한 데이터를 사용한 기술적 분석이다. 데이터가 직접 보여주는 '표시과목 기관 수 구성 변화'의 범위 안에서만 해석하며, 원인·인과, 진료량·접근성, 의료 공급 부족 여부는 다루지 않는다.

## 입력과 출력

17개 분기 Excel은 data/raw/전국_기관수/에 배치. 보조 입력과 실행 순서는 notebooks/README.md 참고.

가공 패널은 data/processed/, 집계표와 그림은 results/EDA/에 저장. 공개용 표, 그림과 검증 범위는 docs/validation.md 참고.

## 전체 수행 흐름

분기 Excel 로드 → 열 통합과 합계 점검 → 전국, 시도, 자치구 비교 → 정의 민감도 → 보조 자료 검토.

## 데이터
| 항목 | 내용 |
|---|---|
| 데이터 | 심평원 지역별 의원 표시과목별 기관 수 (분기별 xlsx 17개) |
| 기간 | 2022Q2 ~ 2026Q2 (17개 분기) |
| 지역 단위 | 17개 시도 · 시군구 |
| 표시과목 | 26종(분기별 등장·소멸 과목 포함) |
| 산출물 | 시군구 long 패널(CSV, 후속 Tableau 입력) · 분석 결과표(CSV) · 그림(PNG) |

분기마다 (1) 표시과목 컬럼 수가 다르고(응급의학과 2024년 등장, 핵의학과 후반 소멸), (2) 시도명이 개편되며(강원도→강원특별자치도, 전라북도→전북특별자치도), (3) 결측·영이 문자 `-`로 입력되어 있다. 이 세 가지를 로딩 단계에서 표준화한다.

In [ ]:
# 설정 · 경로 · 폰트
from pathlib import Path
import re, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
warnings.filterwarnings("ignore")

def find_project_root():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if list(base.glob("data/raw/전국_기관수/기관수현황_지역별의원표시과목별_*.xlsx")):
            return base
    return here

PROJECT_ROOT = find_project_root()
DATA_DIR = next(d for d in (PROJECT_ROOT / "data" / "raw").iterdir()
                if d.is_dir() and list(d.glob("기관수현황_지역별의원표시과목별_*.xlsx")))
PROC_DIR = PROJECT_ROOT/"data"/"processed"
FIG_DIR  = PROJECT_ROOT/"results"/"EDA"/"figures"
TAB_DIR  = PROJECT_ROOT/"results"/"EDA"/"tables"
for d in (PROC_DIR, FIG_DIR, TAB_DIR): d.mkdir(parents=True, exist_ok=True)

def setup_korean_font():
    for fp in ("C:/Windows/Fonts/malgun.ttf",
               "/System/Library/Fonts/AppleSDGothicNeo.ttc",
               "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"):
        if Path(fp).exists():
            font_manager.fontManager.addfont(fp)
            name = font_manager.FontProperties(fname=fp).get_name()
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False
setup_korean_font()

# 색: 감소(적)·증가(청)·중립(회)·서울 강조(주황)
C_NEG, C_POS, C_NEU, C_HI = "#c0392b", "#1f6feb", "#9aa0a6", "#e67e22"
plt.rcParams.update({"figure.dpi":110, "savefig.dpi":150,
                     "axes.grid":True, "grid.alpha":0.25, "axes.axisbelow":True})
short = lambda s: (s.replace("특별시","").replace("광역시","")
                    .replace("특별자치도","").replace("특별자치시","").replace("도",""))
Q0, Q1 = "2022Q2", "2026Q2"
print("DATA_DIR:", DATA_DIR.name, "| xlsx", len(list(DATA_DIR.glob('*.xlsx'))), "개")

## 1. 데이터 로딩과 품질 검증
17개 분기 파일을 하나의 패널로 통합하면서 표시과목 컬럼 합집합 처리, 시도명 표준화, `-`→0 변환을 적용한다. 통합 결과가 원자료와 어긋나지 않는지 **표시과목 합계 = '계' 열** 일치, 시도·서울 자치구 수, 중복 여부로 검증한다.

In [ ]:
# 로드 · 정제 · 통합
SIDO_NORM = {"강원도":"강원특별자치도", "전라북도":"전북특별자치도"}
def quarter_key(fn):
    m = re.search(r"(\d{4})년(\d)분기", fn); y, q = int(m.group(1)), int(m.group(2))
    return f"{y}Q{q}", y*10+q

files = sorted(DATA_DIR.glob("*.xlsx")); raw = []
for f in files:
    d = pd.read_excel(f, header=2).rename(columns=lambda c: str(c).strip())
    d["시도"] = d["시도"].replace(SIDO_NORM)
    q, qn = quarter_key(f.name); d["분기"], d["분기순"] = q, qn
    raw.append(d)

ID = ["분기","분기순","기준분기","시도","시군구","계"]; SUBJECTS = []
for d in raw:
    for c in d.columns:
        if c not in ID and c not in SUBJECTS: SUBJECTS.append(c)

def clean(d):
    d = d.copy()
    for c in SUBJECTS:
        if c not in d.columns: d[c] = 0
    d = d[["분기","분기순","시도","시군구","계"]+SUBJECTS]
    for c in SUBJECTS+["계"]:
        d[c] = pd.to_numeric(d[c].replace("-",0), errors="coerce").fillna(0).astype(int)
    return d

wide = pd.concat([clean(d) for d in raw], ignore_index=True)
sido_wide = wide[wide["시군구"]=="계"].reset_index(drop=True)
sgg_wide = wide[(wide["시군구"]!="계") & wide["시군구"].notna()].reset_index(drop=True)
sgg_wide = sgg_wide[sgg_wide["시군구"].astype(str).str.endswith(("구","시","군"))].reset_index(drop=True)

# 후속 Tableau 입력용 tidy long 패널 저장
long = sgg_wide.melt(id_vars=["분기","분기순","시도","시군구"], value_vars=SUBJECTS,
                     var_name="표시과목", value_name="기관수")
long.to_csv(PROC_DIR/"의원_표시과목_시군구_long.csv", index=False, encoding="utf-8-sig")

# 품질 검증
chk = wide.copy(); chk["과목합"] = chk[SUBJECTS].sum(axis=1)
mismatch = int((chk["계"] != chk["과목합"]).sum())
seoul_gu = sgg_wide.loc[sgg_wide["시도"]=="서울특별시","시군구"].nunique()
dup = int(sgg_wide.duplicated(["분기","시도","시군구"]).sum())
assert mismatch==0 and seoul_gu==25 and dup==0
print(f"통합 wide {wide.shape} | 표시과목 {len(SUBJECTS)}종 | 분기 {wide['분기순'].nunique()}개")
print(f"시도 {sido_wide['시도'].nunique()} | 서울 자치구 {seoul_gu} | 시군구 행(Q1) {len(sgg_wide[sgg_wide['분기']==Q1])}")
print(f"품질검증 → 계=과목합 불일치 {mismatch}, 중복 {dup} : 통과")

### 결과 해석
17개 분기·17개 시도·시군구가 하나의 패널로 통합되었고, 모든 행에서 표시과목 합계가 '계' 열과 정확히 일치했다(불일치 0, 중복 0, 서울 자치구 25개 확인). 이후 분석은 이 통합 패널을 사용하며, 시군구 단위 long 패널은 후속 Tableau 시각화의 입력으로 저장했다.

## 2. 전국 의원 총량 변화
### 분석 목적
개별 표시과목의 변화를 해석하기 위한 기준선으로, 전국 의원 총수의 추세를 먼저 확인한다.
### 분석 방법
분기별 시도 '계'를 합산해 전국 의원 총수의 17개 분기 시계열을 그린다.

In [ ]:
nat = sido_wide.groupby(["분기순","분기"])["계"].sum().reset_index().sort_values("분기순")
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(nat["분기"], nat["계"], marker="o", color=C_POS, lw=2)
ax.set_title("전국 의원 총수 분기 추이 (2022Q2–2026Q2)")
ax.set_xlabel("분기"); ax.set_ylabel("의원 수 (개소)")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout(); fig.savefig(FIG_DIR/"fig_national_trend.png"); plt.show()

g0, g1 = int(nat["계"].iloc[0]), int(nat["계"].iloc[-1])
nat_growth = 100*(g1-g0)/g0
print(f"전국 의원 수 {g0:,} → {g1:,} ({nat_growth:+.1f}%)")

### 결과 해석
전국 의원 수는 17개 분기 동안 감소 없이 꾸준히 증가해 **34,541개소 → 37,731개소, 약 +9.2%** 늘었다. 전체 규모가 늘었다는 사실을 기준선으로 두고, 다음 절에서 이 증가가 표시과목별로 균등했는지 확인한다.

### 배경: 종별 공급 구조에서 의원의 위치 - 부록
#### 분석 목적
본 분석은 '의원(1차 의료)'의 표시과목만 다룬다. 그 전제로, 전체 요양기관 공급에서 의원이 차지하는 위치와 정신병원 등 종별 추세를 배경으로 확인한다.
#### 분석 방법
종별 요양기관 현황(시군구×종)에서 전국 의원 수·정신병원 수 추이를 본다. 이는 **배경·교차검증용**이며, 종별(기관 유형)은 표시과목(진료과 구성)과 다른 축이므로 본 분석의 결론 근거로는 쓰지 않는다.

In [ ]:
# 배경(교차검증): 전국 종별 요양기관 추세 — 의원·정신병원
import re as _re
yg_f = next((q for q in (PROJECT_ROOT / "data" / "raw").rglob("*요양기관*.csv")), None) or next((q for q in (PROJECT_ROOT / "data" / "raw").rglob("*요양기관*.xlsx")), None)
try:
    if yg_f is None:
        raise FileNotFoundError("요양기관 파일 없음")
    d = pd.read_csv(yg_f, encoding="cp949") if yg_f.suffix==".csv" else pd.read_excel(yg_f)
    det = d.iloc[0]
    for _i in range(min(3, len(d))):                 # 세부 종별명(의원·정신병원)이 있는 행 탐색
        if (d.iloc[_i].astype(str).str.strip() == "의원").any():
            det = d.iloc[_i]; break
    key = d.columns[0]
    whole = d[d[key].astype(str).str.contains("전체|전국", na=False)].iloc[[0]]
    def _num(x): return pd.to_numeric(str(x).replace(",",""), errors="coerce")
    qcols = [c for c in d.columns if c!=key]
    def qbase(c):
        m=_re.match(r"(\d{4}\.\d/\d)", str(c)); return m.group(1) if m else None
    rec=[]
    for qb in sorted({qbase(c) for c in qcols if qbase(c)}):
        cols=[c for c in qcols if qbase(c)==qb]
        lab={c:str(det[c]).strip() for c in cols}
        def gv(name):
            for c in cols:
                if lab[c]==name: return _num(whole[c].iloc[0])
            return np.nan
        y,q = qb[:4], qb.split(".")[1].split("/")[0]
        rec.append({"분기":f"{y}Q{q}","분기순":int(y)*10+int(q),
                    "의원":gv("의원"),"정신병원":gv("정신병원")})
    B = pd.DataFrame(rec).sort_values("분기순")
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    ax[0].plot(B["분기"],B["의원"],marker="o",color=C_POS); ax[0].set_title("전국 의원 수 추이(종별 통계)"); ax[0].tick_params(axis="x",rotation=45)
    ax[1].plot(B["분기"],B["정신병원"],marker="s",color=C_HI); ax[1].set_title("전국 정신병원 수 추이"); ax[1].tick_params(axis="x",rotation=45)
    fig.tight_layout(); fig.savefig(FIG_DIR/"fig_bg_jongbyeol.png"); plt.show()
    ui0,ui1=B["의원"].iloc[0],B["의원"].iloc[-1]; ps0,ps1=B["정신병원"].iloc[0],B["정신병원"].iloc[-1]
    print(f"종별 통계 의원 {ui0:.0f}→{ui1:.0f} ({100*(ui1-ui0)/ui0:+.1f}%) | 정신병원 {ps0:.0f}→{ps1:.0f} ({100*(ps1-ps0)/ps0:+.1f}%)")
    print("※ 표시과목 통계의 의원 총계와 대략 일치하는지 교차검증에 사용.")
except Exception as e:
    print("종별 배경 생략:", e)


### 결과 해석
종별 요양기관 통계상 전국 의원 수 추세는 본 분석(표시과목 통계)의 의원 총계와 대체로 일치해 **교차검증** 역할을 한다. 정신병원 수 증가는 뒤에서 확인할 정신건강의학과 '표시과목' 급증과 기관 유형 수준에서 호응한다. 다만 종별(기관 유형)은 표시과목(진료과 구성)과 다른 축이므로 이 절은 배경으로만 두고, 재편·비중 해석의 근거로는 쓰지 않는다.

## 3. 진료과목별 기관 수 변화
### 분석 목적
전체 의원 수가 증가하는 동안 그 증가가 모든 표시과목에서 균등하게 나타났는지 확인한다.
### 분석 방법
기준 시점(2022Q2)과 최종 시점(2026Q2)의 표시과목별 전국 기관 수를 비교해 증감률을 계산한다. 기준 시점 기관 수가 10개소 미만인 소규모·행정계 과목(핵의학과·진단검사의학과 등)은 증감률이 불안정해 그림에서 제외하고, 결과표에는 유지한다.

In [ ]:
nat_subj = sido_wide.groupby("분기순")[SUBJECTS].sum()
first, last = nat_subj.iloc[0], nat_subj.iloc[-1]
gr = pd.DataFrame({Q0:first, Q1:last})
gr["증감(개소)"] = gr[Q1]-gr[Q0]
gr["증감률(%)"] = 100*(gr[Q1]-gr[Q0])/gr[Q0].replace(0, np.nan)
gr.sort_values("증감률(%)", ascending=False).to_csv(TAB_DIR/"과목별_전국_증감률.csv", encoding="utf-8-sig")

gr_plot = gr[gr[Q0]>=10].dropna(subset=["증감률(%)"]).sort_values("증감률(%)")
fig, ax = plt.subplots(figsize=(10,7))
colors = [C_POS if v>=nat_growth else (C_NEG if v<0 else C_NEU) for v in gr_plot["증감률(%)"]]
ax.barh(gr_plot.index, gr_plot["증감률(%)"], color=colors)
ax.axvline(nat_growth, color="#444", ls="--", lw=1)
ax.text(nat_growth, -0.9, f" 전국 평균 {nat_growth:+.1f}%", color="#444", fontsize=9)
ax.set_title("표시과목별 전국 기관 수 증감률 (2022Q2→2026Q2)\n청=평균 이상 · 회=평균 미만 · 적=감소")
ax.set_xlabel("증감률 (%)"); ax.set_ylabel("표시과목")
fig.tight_layout(); fig.savefig(FIG_DIR/"fig_subject_growth.png"); plt.show()

### 결과 해석
전체 평균 +9.2%를 기준으로 표시과목별 증감률의 편차가 크게 나타났다. 증가율 상위는 **신경과 +33.1%, 정신건강의학과 +20.7%, 재활의학과 +19.4%, 정형외과 +15.2%** 였다. 반면 규모가 큰 표시과목 중 **소아청소년과 +0.6%, 가정의학과 +1.6%, 산부인과 +1.3%** 는 사실상 정체했다. 전체 의원 수 증가가 모든 표시과목에서 균등하게 나타나지 않았고, 일부 과목에 증가가 집중되었음을 확인할 수 있다.

### 배경: 전국 소아 인구 급감과 소아청소년과 기관 수
#### 분석 목적
소아청소년과의 정체를 '진료과 선호'만으로 읽지 않도록, 수요 측 배경인 소아(0–14세) 인구 변화를 함께 본다.
#### 분석 방법
전국 인구현황(시군구×연령)에서 전국 소아(0–14세) 인구를 분기말 기준으로 집계하고, 소아청소년과 기관 수와 지수(기준=100)로 비교한다. 두 지표의 **병렬 관찰**이며 접근성·미충족 수요를 측정하지 않는다(인과 해석 금지).

In [ ]:
# 배경: 전국 소아(0-14세) 인구 vs 소아청소년과 기관 수 (병렬 관찰)
import re as _re

# '전국'+'인구' 가 경로(폴더명 포함)에 든 파일을 찾음 — 파일명에 '전국'이 없어도 됨
def _find_nat_pop_files():
    csvs = [q for q in (PROJECT_ROOT / "data" / "raw").rglob("*.csv")
            if ("전국" in str(q)) and ("인구" in str(q)) and ("전처리" not in q.name)]
    if csvs:
        return sorted(set(csvs))
    return sorted({q for q in (PROJECT_ROOT / "data" / "raw").rglob("*.xlsx")
                   if ("전국" in str(q)) and ("인구" in str(q))})

def _num(x):
    return pd.to_numeric(str(x).replace(",", ""), errors="coerce")

def _parse_nat_pop(f):
    d = pd.read_csv(f, encoding="cp949") if f.suffix == ".csv" else pd.read_excel(f)
    d = d.rename(columns={d.columns[0]: "행정구역"})
    nat = d[d["행정구역"].astype(str).str.replace(" ", "").str.startswith("전국")]
    row = (nat.iloc[0] if len(nat) else d.iloc[0])   # 전국 행 없으면 첫 집계 행
    agg = {}
    for c in d.columns:
        m = _re.match(r"(\d{4})년(\d{2})월_계_(.+)", str(c).replace(" ", ""))
        if not m:
            continue
        y, mm, band = m.groups()
        agg.setdefault((y, mm), {"tot": np.nan, "kid": 0.0})
        if band == "총인구수":
            agg[(y, mm)]["tot"] = _num(row[c])
        else:
            rng = _re.match(r"(\d+)[~～\-](\d+)세", band)      # 0~4, 5~9, 10~14 만 합산
            if rng and int(rng.group(2)) <= 14:
                v = _num(row[c])
                if pd.notna(v):
                    agg[(y, mm)]["kid"] += v
    qm = {"03": "Q1", "06": "Q2", "09": "Q3", "12": "Q4"}   # 분기말 월만 사용
    return [{"분기": f"{y}{qm[mm]}", "소아0_14": v["kid"], "총인구": v["tot"]}
            for (y, mm), v in agg.items() if mm in qm and pd.notna(v["tot"])]

try:
    files = _find_nat_pop_files()
    if not files:
        raise FileNotFoundError("전국 인구현황 파일(폴더)을 찾지 못함")
    recs = []
    for f in files:
        try:
            recs += _parse_nat_pop(f)
        except Exception:
            pass
    P = pd.DataFrame(recs).drop_duplicates("분기")
    if P.empty:
        raise ValueError("인구 컬럼(YYYY년MM월_계_총인구수 등)을 읽지 못함 — 컬럼 형식 확인 필요")

    ped = sido_wide.groupby("분기")["소아청소년과"].sum().reset_index()
    J = P.merge(ped, on="분기").sort_values("분기")
    J["소아인구_지수"] = 100 * J["소아0_14"] / J["소아0_14"].iloc[0]
    J["소아과_지수"]   = 100 * J["소아청소년과"] / J["소아청소년과"].iloc[0]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(J["분기"], J["소아인구_지수"], marker="o", color=C_NEG, label="소아 인구(0–14세)")
    ax.plot(J["분기"], J["소아과_지수"],  marker="s", color=C_POS, label="소아청소년과 기관 수")
    ax.axhline(100, color="#999", lw=0.7)
    ax.set_ylabel("지수(기준=100)"); ax.legend(); ax.tick_params(axis="x", rotation=45)
    ax.set_title("전국 소아 인구 vs 소아청소년과 기관 수 — 병렬 관찰")
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_bg_child_pop.png"); plt.show()

    pc = 100 * (J["소아0_14"].iloc[-1]      - J["소아0_14"].iloc[0])      / J["소아0_14"].iloc[0]
    pp = 100 * (J["소아청소년과"].iloc[-1] - J["소아청소년과"].iloc[0]) / J["소아청소년과"].iloc[0]
    print(f"사용 파일 {len(files)}개 · 분기 {J['분기'].nunique()}개")
    print(f"전국 소아(0-14) 인구 {pc:+.1f}% | 소아청소년과 기관 수 {pp:+.1f}% (병렬 관찰)")
except Exception as e:
    print("전국 소아 인구 배경 생략:", e)

### 결과 해석
전국 소아(0–14세) 인구는 크게 감소한 반면, 소아청소년과 기관 수는 상대적으로 완만하게 변한다. 즉 소아청소년과의 정체를 '진료과 선호'만으로 보기 어렵고 **소아 인구 급감이라는 수요 배경과 나란히** 나타난다. 단, 이는 기관 수와 인구의 병렬 관찰이며 접근성·미충족 수요를 측정한 것이 아니다. 따라서 '소아 인구 감소 → 소아과 감소 → 공백'과 같은 인과 서술로 확대하지 않는다.

## 4. 1차의료 핵심과 운영정의
지역 비교의 지표로 '1차의료 핵심과 비중'(핵심과 기관 수 ÷ 전체 의원 수 × 100)을 사용한다. 주 분석에서는 동네의원 일상 외래의 3개 표시과목인 **내과 · 소아청소년과 · 가정의학과**를 핵심과로 정의한다. 가정의학과는 일차의료를 표방하는 전문과이고, 내과·소아청소년과는 의원 외래에서 기관 수가 가장 많은 과로, 세 과는 '동네에서 일상적으로 찾는 1차외래'라는 하나의 개념으로 묶인다.

이 정의는 **본 분석의 운영정의**이며, 정부가 병원급에서 지정·논의하는 '필수의료'와는 다른 개념이다. 산부인과·외과는 분만·시술 중심이라 동네 일차외래와 성격이 달라 주 정의에서 제외하고, 정의 선택이 결과에 미치는 영향은 마지막 민감도 절에서 별도로 확인한다.

| 정의 | 포함 표시과목 | 용도 |
|---|---|---|
| 좁게 | 소아청소년과·산부인과·외과 | 민감도 비교 |
| 기본 | 좁게 + 내과·가정의학과 | 민감도 비교 |
| **주 정의(1차의료 핵심)** | **내과·소아청소년과·가정의학과** | **본 분석 기준** |
| 넓게 | 기본 + 이비인후과·정형외과 | 민감도 비교 |

In [ ]:
MAIN = ["내과", "소아청소년과", "가정의학과"]          # 주 분석 운영정의
DEFS = {"좁게": ["소아청소년과","산부인과","외과"],
        "기본": ["소아청소년과","산부인과","외과","내과","가정의학과"],
        "주 정의": MAIN,
        "넓게": ["소아청소년과","산부인과","외과","내과","가정의학과","이비인후과","정형외과"]}

def sido_ratio(qq, cols):
    g = sido_wide[sido_wide["분기"]==qq].set_index("시도")
    return 100*g[cols].sum(axis=1)/g["계"]

본 프로젝트에서는 1차 의료를 내과, 소아청소년과, 가정의학과로 정의한다.

> **표시과목의 제도적 의미(방법론 주석).** 본 분석의 '표시과목'은 심평원 집계상 개설자가 자기 전문과목을 요양기관 명칭(간판)에 표시한 것을 기준으로 한다(의료법 시행규칙 제41조 '진료과목', 제77조 제2항 '전문과목'과 구분되는 개념). 이는 정부의 병원급 중심 '필수의료' 정의(중증·응급·분만·소아; 2024 의료개혁 1차 실행방안 6대 우선순위)와 **층위가 다르다.** 따라서 '1차의료 핵심(내과+소아+가정)'은 필수의료를 재는 지표가 아니라 **동네의원 일차외래의 간판 구성을 보기 위한 운영정의**다.

## 5. 지역별(17개 시도) 일차의료 핵심과 비중 변화
### 분석 목적
핵심과 비중 변화가 지역별로 어떻게 다른지, 그리고 서울이 다른 시도와 비교해 어느 위치에 있는지 확인한다.
### 분석 방법
시도별로 기준·최종 시점의 핵심과(내과+소아청소년과+가정의학과) 비중을 구하고 그 차이(pp)를 비교한다. 각 시도의 비중은 시도 내 기관 수를 합산해 계산한 값이다.

In [ ]:
sido_ch = pd.DataFrame({name: sido_ratio(Q1,c)-sido_ratio(Q0,c) for name,c in DEFS.items()})
sido_ch.round(2).to_csv(TAB_DIR/"시도별_핵심과비중_정의4종.csv", encoding="utf-8-sig")

ch_main = sido_ch["주 정의"].sort_values()
fig, ax = plt.subplots(figsize=(9,6))
cols = [C_HI if i=="서울특별시" else (C_NEG if v<0 else C_POS) for i,v in ch_main.items()]
ax.barh([short(i) for i in ch_main.index], ch_main.values, color=cols)
ax.axvline(ch_main.mean(), color="#444", ls="--", lw=1)
ax.text(ch_main.mean(), -0.9, f" 시도 평균 {ch_main.mean():+.2f}pp", color="#444", fontsize=9)
ax.set_xlabel("일차의료 핵심과 비중 변화 (%p)"); ax.set_ylabel("시도")
ax.set_title("17개 시도 일차의료 핵심과(내과+소아청소년과+가정의학과) 비중 변화\n(2022Q2→2026Q2, 주황=서울, 점선=시도 평균)")
fig.tight_layout(); fig.savefig(FIG_DIR/"fig_sido_rank.png"); plt.show()

rk = list(ch_main.index).index("서울특별시")+1
print(f"서울 {ch_main['서울특별시']:+.2f}pp = 감소폭 {rk}위/{len(ch_main)} | 2위 {short(ch_main.index[1])} {ch_main.iloc[1]:+.2f}pp")

### 결과 해석
주 정의 기준으로 **서울의 핵심과 비중 감소가 −1.12%p로 17개 시도 중 가장 컸고, 경기(−0.95%p)가 그 뒤를 이어** 수도권 두 지역이 최상위 감소 쌍을 이뤘다. 반대로 경상북·경상남·강원 등 일부 시도는 비중이 소폭 증가했다. 즉 핵심과 비중 감소는 전국에서 균일하지 않았고, 주 정의 기준으로는 서울에서 가장 두드러졌다. 이 순위가 정의 선택에 얼마나 견고한지는 8절에서 확인한다.

## 6. 지역유형별 비교
### 분석 목적
'지방(군 지역)일수록 핵심과 비중 감소가 크다'는 통념이 데이터와 일치하는지 확인한다.
### 분석 방법
시군구를 도시성 기준 4유형(서울 / 광역시 구 / 도-시·구 / 군)으로 나눈다. '도' 안의 시·구와 군을 섞으면 군 지역의 패턴이 가려지므로 군을 별도 유형으로 분리한다. 각 유형의 비중은 **유형 내 기관 수를 모두 합산한 뒤 계산**한 값(가중 비중)이며, 시군구별 비중의 단순 평균이 아니다.

In [ ]:
def urban_type(sido, g):
    if sido=="서울특별시": return "서울"
    if sido.endswith("광역시") or sido=="세종특별자치시": return "군" if g.endswith("군") else "광역시"
    return "군" if g.endswith("군") else "도-시·구"

sgg_wide["도시유형"] = [urban_type(s,g) for s,g in zip(sgg_wide["시도"], sgg_wide["시군구"])]
order = ["서울","광역시","도-시·구","군"]
def type_ratio(qq):
    g = sgg_wide[sgg_wide["분기"]==qq]
    return g.groupby("도시유형").apply(lambda x: 100*x[MAIN].sum().sum()/x["계"].sum())

rt = pd.DataFrame({Q0:type_ratio(Q0), Q1:type_ratio(Q1)}).reindex(order)
rt["변화(%p)"] = rt[Q1]-rt[Q0]
rt["n(시군구)"] = sgg_wide[sgg_wide["분기"]==Q1].groupby("도시유형").size().reindex(order)
rt.round(2).to_csv(TAB_DIR/"지역유형별_핵심과비중.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8,4.5))
cc = [C_HI if k=="서울" else (C_NEG if v<0 else C_POS) for k,v in rt["변화(%p)"].items()]
ax.bar(rt.index, rt["변화(%p)"], color=cc)
for i,v in enumerate(rt["변화(%p)"]):
    ax.text(i, v+(0.03 if v>=0 else -0.09), f"{v:+.2f}", ha="center", fontsize=9)
ax.axhline(0, color="#444", lw=0.8)
ax.set_ylabel("일차의료 핵심과 비중 변화 (%p)"); ax.set_xlabel("지역유형")
ax.set_title("지역유형별 일차의료 핵심과 비중 변화 (2022Q2→2026Q2)\n(기관 수 합산 기준 가중 비중)")
fig.tight_layout(); fig.savefig(FIG_DIR/"fig_urban_type.png"); plt.show()
display(rt.round(2))

### 결과 해석
비중 변화는 **서울 −1.12%p, 도-시·구 −0.34%p, 광역시 −0.27%p, 군 +0.21%p**로 나타났다(군 n=81). 군 지역은 4유형 중 유일하게 소폭 증가해 사실상 변화가 없었고, 감소는 서울에서 가장 컸다. 이는 '지방(군)일수록 크다'는 통념과 반대 방향이며, 군을 별도 유형으로 분리해야 드러난다. 다만 광역시(−0.27%p)가 도-시·구(−0.34%p)보다 감소가 작아 '도시성이 높을수록 단조적으로 감소가 커진다'고 일반화할 수는 없다. 관찰되는 것은 서울의 감소가 가장 크고 군은 감소하지 않았다는 두 끝의 대비다.

## 7. 서울 상세 — 비중 변화의 구조
### 분석 목적
서울의 핵심과 비중 하락이 핵심과 기관 수 자체의 감소에서 비롯됐는지, 아니면 전체 의원 수 증가에 따른 분모 확대에서 비롯됐는지 구분한다.
### 분석 방법
서울과 전국의 표시과목별 증감률을 나란히 비교하고, 핵심과 3과의 절대 기관 수 변화를 함께 확인한다.

In [ ]:
def subj_first_last(df):
    g = df.groupby("분기순")[SUBJECTS].sum(); return g.iloc[0], g.iloc[-1]
nf, nl = subj_first_last(sido_wide)
sf, sl = subj_first_last(sido_wide[sido_wide["시도"]=="서울특별시"])
cmp = pd.DataFrame({"전국%": 100*(nl-nf)/nf.replace(0,np.nan),
                    "서울%": 100*(sl-sf)/sf.replace(0,np.nan),
                    "서울_2022Q2": sf, "서울_증감(개소)": sl-sf})
cmp = cmp[cmp["서울_2022Q2"]>=50].sort_values("서울%")
cmp.round(1).to_csv(TAB_DIR/"서울vs전국_과목별증감.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(9,7)); y = np.arange(len(cmp))
ax.hlines(y, cmp["전국%"], cmp["서울%"], color=C_NEU, lw=2, zorder=1)
ax.scatter(cmp["전국%"], y, color=C_NEU, label="전국", zorder=2)
ax.scatter(cmp["서울%"], y, color=C_HI, label="서울", zorder=2)
ax.set_yticks(y); ax.set_yticklabels(cmp.index)
ax.axvline(0, color="#444", lw=0.6)
ax.set_xlabel("증감률 (%)"); ax.set_ylabel("표시과목"); ax.legend()
ax.set_title("서울 vs 전국 표시과목별 증감률 (2022Q2→2026Q2)\n(기준 시점 서울 50개소 이상 과목)")
fig.tight_layout(); fig.savefig(FIG_DIR/"fig_seoul_vs_nat.png"); plt.show()

st0 = int(sido_wide[sido_wide['시도']=='서울특별시'].groupby('분기순')['계'].sum().iloc[0])
st1 = int(sido_wide[sido_wide['시도']=='서울특별시'].groupby('분기순')['계'].sum().iloc[-1])
print(f"서울 총수 {st0:,} → {st1:,} ({100*(st1-st0)/st0:+.1f}%) | 전국 {nat_growth:+.1f}%")
for c in MAIN:
    print(f"  {c} {int(sf[c])} → {int(sl[c])} ({100*(sl[c]-sf[c])/sf[c]:+.1f}%)")

### 결과 해석
서울 의원 총수는 **9,359 → 10,634개소(+13.6%)** 로 전국 평균(+9.2%)보다 빠르게 늘었다. 주 정의 3과의 절대 기관 수는 내과 +10.7%, 가정의학과 +5.9%로 늘었고 소아청소년과만 −1.5%로 감소했다. 반면 정신건강의학과(+25.1%)·성형외과(+13.4%)·신경과(+60.4%) 등 핵심과에 포함되지 않는 표시과목이 전국보다 빠르게 증가했다. 따라서 서울의 핵심과 **비중** 하락은 핵심과 자체의 감소가 아니라 전체 의원 수가 더 빠르게 늘어난 데 따른 **상대적 희석** 이 주된 구조다. 이는 진료과 구성의 재편으로 읽을 수 있으며, 데이터는 진료량이나 의료 공급 수준을 측정하지 않으므로 그 이상으로 해석하지 않는다.

여기서 말하는 핵심과는 1차 의료에 해당하는 과이다.

### 서울 25개 자치구 편차
#### 분석 목적
서울 내부에서 핵심과 비중 변화가 자치구별로 얼마나 갈리는지 확인한다.
#### 분석 방법
서울 25개 자치구별로 기준·최종 시점의 핵심과 비중을 구하고 그 변화량(pp)을 비교한다.

In [ ]:
seoul = sgg_wide[sgg_wide["시도"]=="서울특별시"]
def gu_ratio(qq):
    g = seoul[seoul["분기"]==qq].set_index("시군구"); return 100*g[MAIN].sum(axis=1)/g["계"]
gu = pd.DataFrame({Q0:gu_ratio(Q0), Q1:gu_ratio(Q1)})
gu["변화(%p)"] = gu[Q1]-gu[Q0]; gu = gu.sort_values("변화(%p)")
gu.round(2).to_csv(TAB_DIR/"서울자치구_핵심과비중.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(9,8))
ax.barh(gu.index, gu["변화(%p)"], color=[C_NEG if v<0 else C_POS for v in gu["변화(%p)"]])
ax.axvline(0, color="#444", lw=0.8)
ax.axvline(gu["변화(%p)"].median(), color=C_HI, ls="--", lw=1, label=f"중앙값 {gu['변화(%p)'].median():+.2f}%p")
ax.legend(); ax.set_xlabel("1차의료 핵심과 비중 변화 (%p)"); ax.set_ylabel("자치구")
ax.set_title("서울 25개 자치구 1차의료 핵심과 비중 변화 (2022Q2→2026Q2)\n(적=하락 · 청=상승)")
fig.tight_layout(); fig.savefig(FIG_DIR/"fig_seoul_gu.png"); plt.show()

nd, nu = int((gu["변화(%p)"]<0).sum()), int((gu["변화(%p)"]>0).sum())
print(f"하락 {nd}구 / 상승 {nu}구 | 범위 {gu['변화(%p)'].min():+.2f}~{gu['변화(%p)'].max():+.2f}%p")
print("최대 하락:", ", ".join(f"{i}({v:+.2f})" for i,v in gu['변화(%p)'].head(3).items()))

### 결과 해석
서울 25개 자치구 중 **20개 구에서 핵심과 비중이 하락했고 5개 구에서 상승**했다. 변화 범위는 −2.50 ~ +1.03%p(중앙값 −0.56%p)로 자치구 간 편차가 존재한다. 하락 폭이 큰 자치구는 용산구(−2.50%p)·중구(−2.19%p)·송파구(−2.18%p)였다. 하락이 큰 자치구는 강남권과 도심·주거지가 섞여 있어 특정 권역에 공간적으로 집중되지 않는다. 자치구 간 편차 자체는 후속 지도(코로플레스) 시각화로 확인할 값이 있으나, 본 데이터만으로 그 편차의 원인을 규정하지는 않는다.

### 서울 자치구 인구 맥락 — 소아 인구비중과의 병렬 관찰
#### 분석 목적
서울 자치구별 일차의료 핵심과 비중 변화 옆에 소아(0–14세) 인구비중을 나란히 놓아, 자치구 지도(대시보드)에 인구 맥락 레이어를 더한다.
#### 분석 방법
서울 자치구 연령인구 전처리본(자치구×분기)에서 소아·고령 인구비중을 시점별로 구해 앞의 자치구 핵심과 비중(`gu`)에 결합하고, 소아 인구비중과 핵심과 비중 변화의 관계를 상관으로 확인한다. 이는 두 지표의 **병렬 관찰**이며 접근성·미충족 수요를 측정하지 않는다(인과 해석 금지).

In [ ]:
# 서울 자치구: 핵심과 비중 × 연령 인구 결합 (대시보드 보조 레이어)
pop_f = next((q for q in (PROJECT_ROOT / "data" / "raw").rglob("*인구_전처리*.csv")), None)
if pop_f is None:
    print("서울 인구현황_전처리 CSV를 찾지 못함 → 이 절 생략(핵심 결과 영향 없음).")
else:
    pop = pd.read_csv(pop_f, encoding="utf-8-sig")
    kid = pop[pop["분기"].isin([Q0,Q1])].pivot_table(index="시군구", columns="분기", values="소아비중%")
    old = pop[pop["분기"].isin([Q0,Q1])].pivot_table(index="시군구", columns="분기", values="고령비중%")
    tot = pop[pop["분기"]==Q1].set_index("시군구")["총인구"]

    M = gu.rename(columns={Q0:"핵심과비중_"+Q0, Q1:"핵심과비중_"+Q1, "변화(%p)":"핵심과비중_변화%p"}).copy()
    M["소아비중_"+Q1]   = kid[Q1]
    M["소아비중_변화%p"] = kid[Q1]-kid[Q0]
    M["고령비중_"+Q1]   = old[Q1]
    M["총인구_"+Q1]     = tot
    M.round(3).to_csv(TAB_DIR/"서울자치구_핵심과_인구_통합.csv", encoding="utf-8-sig")

    r = M["핵심과비중_변화%p"].corr(M["소아비중_"+Q1])
    fig, ax = plt.subplots(figsize=(7.5,5.5))
    ax.scatter(M["소아비중_"+Q1], M["핵심과비중_변화%p"], color=C_POS, zorder=3)
    for g,row in M.iterrows():
        ax.annotate(g, (row["소아비중_"+Q1], row["핵심과비중_변화%p"]), fontsize=8, alpha=0.7)
    ax.axhline(0, color="#444", lw=0.6)
    ax.set_xlabel(f"자치구 소아(0–14세) 인구비중 (%, {Q1})")
    ax.set_ylabel("일차의료 핵심과 비중 변화 (%p)")
    ax.set_title(f"서울 자치구: 소아 인구비중 vs 핵심과 비중 변화 — 병렬 관찰 (r={r:+.2f})")
    fig.tight_layout(); fig.savefig(FIG_DIR/"fig_seoul_gu_pop.png"); plt.show()

    dk = M["소아비중_변화%p"]
    print(f"자치구 소아 인구비중 변화 범위 {dk.min():+.2f}~{dk.max():+.2f}%p | 하락 {int((dk<0).sum())}구 / 상승 {int((dk>0).sum())}구")
    print(f"핵심과 비중 변화 vs 소아 인구비중 상관 r = {r:+.2f} (n={len(M)})")


### 결과 해석
서울 25개 자치구의 소아(0–14세) 인구비중은 **예외 없이 전역에서 하락**했다(대체로 −0.6 ~ −1.7%p). 따라서 특정 자치구의 핵심과 비중 변화를 소아 인구 감소의 결과로 단정할 수 없다. 핵심과 비중 변화와 자치구 소아 인구비중의 상관은 위 코드 출력의 `r`로 확인하며, 소아 인구가 전역에서 함께 감소한 구조상 이 관계를 인과로 읽지 않는다.

## 8. 정의 민감도 - 부록
### 분석 목적
'서울의 핵심과 비중 감소가 가장 크다'는 결과가 핵심과 정의 선택에 얼마나 좌우되는지 확인한다.
### 분석 방법
네 가지 정의(좁게 / 기본 / 주 정의 / 넓게)별로 서울의 시도 내 감소폭 순위를 비교한다.

In [ ]:
rows = []
for name, cols in DEFS.items():
    ch = (sido_ratio(Q1,cols)-sido_ratio(Q0,cols)).sort_values()
    rk = list(ch.index).index("서울특별시")+1
    rows.append({"정의":name, "포함 과목수":len(cols), "서울 변화(%p)":round(ch["서울특별시"],2),
                 "서울 순위":rk, "1위 시도":short(ch.index[0]), "1위 변화(%p)":round(ch.iloc[0],2)})
sens = pd.DataFrame(rows); sens.to_csv(TAB_DIR/"정의별_서울순위.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8,4.5))
order_def = ["좁게","기본","주 정의","넓게"]
sens_o = sens.set_index("정의").reindex(order_def)
bars = ax.bar(order_def, sens_o["서울 변화(%p)"],
              color=[C_HI if d=="주 정의" else C_NEU for d in order_def])
for i,(v,r) in enumerate(zip(sens_o["서울 변화(%p)"], sens_o["서울 순위"])):
    ax.text(i, v+0.10, f"{v:+.2f}%p\n순위 {int(r)}/17", ha="center", va="bottom",
            fontsize=9, color="white", fontweight="bold")
ax.axhline(0, color="#444", lw=0.8)
ax.set_ylim(bottom=float(sens_o["서울 변화(%p)"].min())-0.35, top=0.05)
ax.set_ylabel("서울 핵심과 비중 변화 (%p)"); ax.set_xlabel("핵심과 정의")
ax.set_title("정의별 서울 핵심과 비중 변화와 시도 내 순위\n(주황=주 분석 정의)")
fig.tight_layout(); fig.savefig(FIG_DIR/"fig_definition_sensitivity.png"); plt.show()
display(sens)

### 결과 해석
서울의 감소폭 순위는 정의에 따라 달라진다. 내과·가정의학과를 포함하는 정의에서 서울이 최상위권으로, **주 정의에서 1위(−1.12%p), 넓게 정의에서도 1위(−1.55%p)**, 기본 정의에서는 2위(−1.58%p, 1위 광주 −1.71%p)였다. 반면 내과·가정의학과를 뺀 **좁은 정의(소아청소년과+산부인과+외과)에서는 서울이 6위(−1.11%p, 1위 광주 −1.39%p)** 로 내려간다. 즉 '서울 최대 감소'라는 결과는 일차외래 중심의 주 정의에 의존하며, 정의를 좁히면 성립하지 않는다. 본 분석은 주 정의를 기준으로 서울을 상세 분석 대상으로 삼되, 이 결과가 정의 선택에 조건부라는 점을 함께 명시한다.

## 9. 서울 소아 야간·응급 의료체계 배치
### 분석 목적
앞 절들에서 확인한 '소아청소년과 표시과목 정체'를 전달체계 배치 측면에서 보조적으로 살핀다. 신규 데이터(서울시 소아의료체계 구축)는 서울의 야간·응급 소아진료 기관 위치와 진료 수준을 담고 있어, 기관 수 구성 변화와는 다른 축인 **'어떤 수준의 소아진료가 어느 자치구에 배치되어 있는가'** 를 기술한다.
### 분석 방법
원자료는 (1) 동일 기관이 테마·콘텐츠코드 중복으로 여러 행 등장, (2) 진료범위 표기 흔들림·문자 깨짐(예: '중증?응급'), (3) 구명칭 결측이 있어 로딩 단계에서 표준화한다. 기관명 기준으로 중복을 제거해 고유 기관 단위로 축약하고, 진료범위를 중증·응급 / 준응급 / 경증(야간외래) 3단계로 정규화한 뒤 진료단계별·자치구별 분포와 24시간 운영 여부, 25개 자치구 커버리지를 집계한다. 기관 수는 접근성·진료량·미충족 수요를 측정하지 않는 **병렬 관찰**이다.

In [ ]:
soa_f = next((q for q in (PROJECT_ROOT / "data" / "raw").rglob("*소아의료체계*.csv")), None)
if soa_f is None:
    print("서울 소아의료체계 CSV를 찾지 못함 → 이 절 생략.")
else:
    df = pd.read_csv(soa_f, encoding="cp949")
    # 1) 구명칭 결측 → 도로명주소에서 '○○구' 추출 보정
    def _gu(a):
        m = re.search(r"서울특별시\s+(\S+구)", str(a)); return m.group(1) if m else None
    df["자치구"] = df["구명칭"].astype(str).str.strip().replace("", np.nan)
    df["자치구"] = df["자치구"].fillna(df["도로명주소"].map(_gu))
    # 2) 진료범위 정규화(표기 흔들림·깨진 문자 보정) → 3단계
    def _tier(x):
        x = str(x).replace("?", "·").lstrip("- ").strip()
        if x.startswith("중증"): return "중증·응급"
        if x.startswith("준응급"): return "준응급"
        if "경증" in x: return "경증(야간외래)"
        return "기타"
    df["진료단계"] = df["진료범위"].map(_tier)
    df["24시간"]  = df["운영시간"].astype(str).str.contains("24시간")
    # 3) 기관 단위 축약(동일 기관명 중복 제거)
    inst = df.drop_duplicates("콘텐츠명").copy()
    tier_order = ["중증·응급", "준응급", "경증(야간외래)"]
    cov = inst["자치구"].nunique()
    seoul25 = ["종로구","중구","용산구","성동구","광진구","동대문구","중랑구","성북구","강북구","도봉구",
               "노원구","은평구","서대문구","마포구","양천구","강서구","구로구","금천구","영등포구","동작구",
               "관악구","서초구","강남구","송파구","강동구"]
    missing = [g for g in seoul25 if g not in set(inst["자치구"].dropna())]

    fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.8))
    tc = inst["진료단계"].value_counts().reindex(tier_order).fillna(0)
    ax[0].bar(tier_order, tc.values, color=[C_NEG, C_HI, C_POS])
    for i, v in enumerate(tc.values): ax[0].text(i, v+0.3, f"{int(v)}", ha="center", fontsize=10)
    ax[0].set_title("진료단계별 기관 수 (중복 제거)"); ax[0].set_ylabel("기관 수")
    gc = inst["자치구"].value_counts()
    ax[1].barh(gc.index[::-1], gc.values[::-1], color=C_POS)
    ax[1].set_title(f"자치구별 기관 수 ({cov}/25개 구 보유)"); ax[1].set_xlabel("기관 수")
    fig.suptitle("서울 소아 야간·응급 의료체계 배치 — 신규 데이터 EDA", fontweight="bold")
    fig.tight_layout(); fig.savefig(FIG_DIR/"fig_soa_system.png"); plt.show()

    out = inst[["콘텐츠명","자치구","진료단계","24시간","운영시간","도로명주소"]].sort_values(["진료단계","자치구"])
    out.to_csv(TAB_DIR/"서울_소아의료체계_기관목록.csv", index=False, encoding="utf-8-sig")
    print(f"원자료 {len(df)}행 → 중복 제거 후 고유기관 {inst['콘텐츠명'].nunique()}개")
    print("진료단계:", ", ".join(f"{k} {int(tc[k])}개" for k in tier_order))
    print(f"24시간 운영 {int(inst['24시간'].sum())}개 · 야간/시간제 {int((~inst['24시간']).sum())}개")
    print(f"자치구 커버리지 {cov}/25 · 미보유 {missing}")

### 결과 해석
원자료 162행에는 동일 기관이 중복 등장하므로 기관명 기준으로 축약하면 **고유 기관 39개**가 남는다. 이들은 진료 수준에 따라 **중증·응급 3개, 준응급 8개, 경증 야간외래 28개**의 3단계로 나뉜다. 상위 두 단계(11개)는 모두 24시간 병원, 경증 단계(28개)는 야간·시간제 외래로, 24시간 운영 기관과 야간 외래가 명확히 구분된다. 공간적으로는 **25개 자치구 중 22개 구가 최소 1개 기관을 보유**하고 광진구·도봉구·양천구 3개 구는 보유 기관이 없다. 이 EDA는 앞 절의 소아청소년과 표시과목 정체를 야간·응급 전달체계 배치와 나란히 보는 **보조 레이어**이며, 기관 수는 접근성·진료량·미충족 수요를 뜻하지 않는다(병렬 관찰).

> **데이터 주의.** 진료범위 표기가 '중증·응급/준응급/경증'으로 흔들리고 일부 문자가 깨져 있으며(예: '중증?응급'), 구명칭이 빈 행이 있어 도로명주소에서 자치구를 보정했다. 콘텐츠코드도 기관당 복수로 부여된 경우가 있어 기관명 기준으로 축약했다. 미보유 3개 구는 인접 자치구 기관으로 실제 접근이 가능할 수 있어, '공백'이 아니라 '기관 미배치'로만 해석한다.

## 10. 핵심 결과 요약
- 전국 의원 수는 약 4년간 +9.2% 증가했으나 표시과목별 증가율은 크게 갈렸다(신경과·정신건강의학과 등 급증, 소아청소년과·가정의학과·산부인과 정체).
- 주 정의(내과+소아청소년과+가정의학과) 기준으로 일차의료 핵심과 비중 감소는 서울에서 가장 컸고 경기가 뒤를 이었다.
- 지역유형별로는 군 지역이 유일하게 감소하지 않아, '지방일수록 크다'는 통념과 반대 방향이었다.
- 서울의 비중 하락은 핵심과 감소가 아니라 전체 의원 수의 더 빠른 증가에 따른 상대적 희석이 주된 구조였다.
- 서울 25개 자치구 중 20개 구에서 비중이 하락했으나, 하락이 큰 자치구는 공간적으로 한 권역에 집중되지 않았다.
- '서울 최대 감소'는 내과·가정의학과를 포함하는 정의에서 성립하며, 좁은 정의에서는 서울 순위가 6위로 내려간다.

## 분석의 한계
본 분석은 표시과목별 기관 수의 구성 변화에 대한 기술적 분석이며, 다음을 측정하지 않는다.
- **표시과목 ≠ 실제 진료**: 표시과목은 간판 기준이며 실제 진료 행태·수요와 다를 수 있다.
- **기관 수 ≠ 공급량**: 의사 수·진료량·병상 수를 반영하지 않으므로 기관 수 변화를 의료 공급량 변화로 볼 수 없다.
- **수요 축 없음**: 인구·질병 등 수요 지표를 포함하지 않으므로 의료 수요 대비 공급의 충분성은 판단 대상이 아니다.
- **원인 미규명**: 지역 비교는 원자료 집계 비교이며, 공통 요인을 통제하거나 변화의 원인을 규명하지 않는다.
- **행정구역 개편**: 시도명은 표준화했으나 시군구 경계는 시점 간 완전히 동일하지 않다.

## 방법론 한계
- **표시과목 ≠ 실제 기능.** 표시과목은 개설자가 간판에 표시한 전문과목이며 진료 기능·수요를 뜻하지 않는다. 예컨대 산부인과 표시과목 기관 수는 정체이지만, 외부 통계상 분만 가능 의원은 2014→2024년 약 −52.7%로 '기능'은 크게 줄었다(간판과 기능의 괴리). 본 분석은 기능·접근성을 측정하지 않는다.
- **기관 수 기준.** 의사 수·진료량·병상 미반영. 기관 수 변화 ≠ 공급량 변화.
- **인구·종별 배경은 병렬 관찰.** 연령 인구·종별 요양기관 지표는 맥락 제공용이며, 상관이 있어도 인과·미충족 수요 해석에 사용하지 않는다.
- **지역 비교는 원자료 집계 비교.** 공통 요인을 통제한 분석이 아니다. 본 분석의 서울 관련 서술 범위는 '1차의료 핵심 정의 기준 서울의 비중 감소폭이 시도 중 가장 크게 나타났다'이며, '서울이 지방보다 심각/지방 위기와 반대'로 확대하지 않는다.

# 최종 완료 확인

Python 3.14.5의 새 커널에서 원본 ZIP 입력으로 전체 셀 순차 실행 통과. 표기 기준이 다른 과거 결과표와 Tableau 화면 차이는 docs/validation.md에 기록. 공개본에는 실행 출력을 제외.